# VQA-RAD Data Exploration

**STAT GR5293 GenAI — Member 1, Week 1–2 deliverable**

This notebook documents the structure and statistics of VQA-RAD, the medical VQA dataset our project uses. The output is a `DATA_CARD.md` summary plus inline visualizations.

**What this notebook does:**

1. Load VQA-RAD from HuggingFace (`flaviagiammarino/vqa-rad`)
2. Verify train/test splits match published values
3. Classify questions into closed/open per our convention
4. Plot question and answer length distributions
5. Inspect sample images (resolution, aspect ratio, modality)
6. Check train/test image overlap (the official split is by question, not by image)

**Runtime:** ~3 minutes on Colab CPU (no GPU needed). The dataset is ~85 MB.

## 1. Setup

In [ ]:
%pip install -q datasets pandas matplotlib seaborn

In [ ]:
import os, sys
PROJECT_DIR = '/content/peft-medical-vqa'
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f'{PROJECT_DIR} not found. Upload peft-medical-vqa.zip and unzip it,\n'
        'or git clone the repo to that path.'
    )
%cd {PROJECT_DIR}
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
from src.data.load_vqarad import load_vqarad, classify_question_type, split_statistics
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 80)

## 2. Load the dataset

In [ ]:
ds = load_vqarad()
print(ds)

In [ ]:
# Per-split statistics
stats = split_statistics(ds)
stats_df = pd.DataFrame(stats).T
stats_df['closed_pct'] = (stats_df['closed'] / stats_df['total'] * 100).round(1)
stats_df['open_pct']   = (stats_df['open']   / stats_df['total'] * 100).round(1)
stats_df

**Sanity check.** The published split sizes are 1,797 / 451. Our loaded numbers should match exactly. The closed/open ratio is approximately 40/60.

## 3. Sample inspection

In [ ]:
# Show a handful of train examples
samples = []
for i in range(8):
    ex = ds['train'][i]
    samples.append({
        'idx': i,
        'question': ex['question'],
        'answer': ex['answer'],
        'qtype': classify_question_type(ex['answer']),
        'image_size': f"{ex['image'].size[0]}×{ex['image'].size[1]}",
        'mode': ex['image'].mode,
    })
pd.DataFrame(samples)

In [ ]:
# Visualize the first 6 train images with their Q-A
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for i, ax in enumerate(axes.flatten()):
    ex = ds['train'][i]
    ax.imshow(ex['image'], cmap='gray' if ex['image'].mode == 'L' else None)
    ax.set_title(f"Q: {ex['question'][:60]}\nA: {ex['answer'][:50]}",
                 fontsize=9, loc='left')
    ax.axis('off')
plt.suptitle('VQA-RAD train samples', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 4. Question and answer length distributions

These statistics inform our `max_new_tokens` hyperparameter (set to 64 — generous given the actual distribution).

In [ ]:
# Build a tidy DataFrame across both splits
rows = []
for split_name in ['train', 'test']:
    split = ds[split_name]
    for q, a in zip(split['question'], split['answer']):
        rows.append({
            'split': split_name,
            'q_words': len(q.split()),
            'a_words': len(str(a).split()),
            'qtype': classify_question_type(a),
        })
df = pd.DataFrame(rows)
print(df.groupby('split').describe()[['q_words', 'a_words']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data=df, x='q_words', hue='split', bins=30, ax=axes[0],
             multiple='dodge', palette='Set1')
axes[0].set_title('Question length (words)')
axes[0].set_xlabel('Words per question')

sns.histplot(data=df, x='a_words', hue='split', bins=30, ax=axes[1],
             multiple='dodge', palette='Set1')
axes[1].set_title('Answer length (words)')
axes[1].set_xlabel('Words per answer')
axes[1].set_xlim(0, 20)        # 99% of answers are ≤10 words

plt.tight_layout()
plt.show()

# Quantile summary
print('Answer-length quantiles (test split):')
print(df[df.split == 'test']['a_words'].quantile([0.5, 0.9, 0.95, 0.99, 1.0]))

**Key takeaway.** The 99th percentile of answer length on the test split is well under 64 tokens, justifying our choice of `max_new_tokens = 64`.

## 5. Question-type distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, split_name in zip(axes, ['train', 'test']):
    sub = df[df.split == split_name]
    counts = sub.qtype.value_counts()
    ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
           colors=['#4C72B0', '#DD8452'], startangle=90)
    ax.set_title(f'{split_name} (n={len(sub)})')
plt.suptitle('Question type distribution')
plt.tight_layout()
plt.show()

## 6. Image resolution analysis

In [ ]:
img_rows = []
for split_name in ['train', 'test']:
    split = ds[split_name]
    for img in split['image']:
        w, h = img.size
        img_rows.append({
            'split': split_name, 'width': w, 'height': h,
            'aspect': w / h, 'mode': img.mode,
            'pixels': w * h,
        })
img_df = pd.DataFrame(img_rows)
print('Image resolution summary:')
print(img_df.groupby('split')[['width', 'height', 'aspect', 'pixels']].describe().T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(img_df.width, img_df.height, alpha=0.4, s=12)
axes[0].set_xlabel('Width (px)'); axes[0].set_ylabel('Height (px)')
axes[0].set_title('Image dimensions (each dot = one image)')

sns.histplot(img_df.aspect, bins=30, ax=axes[1])
axes[1].set_xlabel('Aspect ratio (W/H)')
axes[1].set_title('Aspect ratio distribution')
axes[1].axvline(1.0, color='red', ls='--', label='square')
axes[1].legend()

mode_counts = img_df['mode'].value_counts()
axes[2].bar(mode_counts.index, mode_counts.values, color='steelblue')
axes[2].set_title('PIL image mode')
axes[2].set_xlabel('mode (L = grayscale, RGB = 3-channel)')

plt.tight_layout()
plt.show()

**Implication for prompting.** Qwen2-VL's Naive Dynamic Resolution handles arbitrary aspect ratios natively, so we do **not** preprocess images. The `VQARadDataset` wrapper does convert grayscale → RGB to keep tensor shapes consistent.

## 7. Train/test image overlap

VQA-RAD's official split is **by question**, not by image. We verify that the same image can appear in both splits with different questions — a known limitation of the benchmark.

In [ ]:
import hashlib

def image_hash(img):
    """MD5 hash of the image pixel bytes — stable identity across splits."""
    return hashlib.md5(img.tobytes()).hexdigest()

train_hashes = {image_hash(img) for img in ds['train']['image']}
test_hashes  = {image_hash(img) for img in ds['test']['image']}
overlap = train_hashes & test_hashes
print(f'Unique images in train: {len(train_hashes)}')
print(f'Unique images in test:  {len(test_hashes)}')
print(f'Overlap:                {len(overlap)} '
      f'({100*len(overlap)/len(test_hashes):.1f}% of test images also seen in train)')
print(f'Total unique images:    {len(train_hashes | test_hashes)}')

**Caveat for the report.** A large fraction of test images is also present in train, just paired with different questions. This is the *standard* VQA-RAD protocol, but it means our test set partially measures generalization to *new questions about familiar images* rather than fully unseen images. We document this explicitly in `docs/DATA_CARD.md` and the final report.

## 8. Summary statistics for the data card

In [ ]:
summary = {
    'train_qa_pairs':     int(len(ds['train'])),
    'test_qa_pairs':      int(len(ds['test'])),
    'total_qa_pairs':     int(len(ds['train']) + len(ds['test'])),
    'unique_images_total': int(len(train_hashes | test_hashes)),
    'unique_images_train': int(len(train_hashes)),
    'unique_images_test':  int(len(test_hashes)),
    'train_test_image_overlap': int(len(overlap)),
    'train_closed_pct': float(stats_df.loc['train', 'closed_pct']),
    'train_open_pct':   float(stats_df.loc['train', 'open_pct']),
    'test_closed_pct':  float(stats_df.loc['test',  'closed_pct']),
    'test_open_pct':    float(stats_df.loc['test',  'open_pct']),
    'q_words_p50': float(df['q_words'].median()),
    'q_words_p99': float(df['q_words'].quantile(0.99)),
    'a_words_p50': float(df['a_words'].median()),
    'a_words_p99': float(df['a_words'].quantile(0.99)),
}
import json
print(json.dumps(summary, indent=2))